### 1. Importação de Bibliotecas e Configurações
Importamos as dependências necessárias e fazemos o reload dos nossos módulos locais (configurações, dataloaders e modelo) para garantir que as alterações mais recentes sejam refletidas.

In [ ]:
import torch.utils.data as data
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import torchvision.transforms as transforms
import torch.optim as optim
import torch
import importlib

import config as cfg
importlib.reload(cfg)

import dataset.dataloader as dl
importlib.reload(dl)

import classifier.cnn as classifier
importlib.reload(classifier)

import utils.metrics as mtcs
importlib.reload(mtcs)

import utils.visualization as vis
importlib.reload(vis)

# Dispositivo de hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Gerar pastas de resultados
cfg.GRAPH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cfg.CNN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

cfg.CNN_VAE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_VAE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_VAE_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

cfg.CNN_GAN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_GAN_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_GAN_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

cfg.CNN_DIFF_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_DIFF_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.CNN_DIFF_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

### 2. Preparação dos Dados e Divisão (Train, Val, Test)
Carregamos os rótulos e aplicamos uma divisão estratificada para garantir que as 75 classes de borboletas estejam devidamente representadas nos conjuntos de treinamento, validação e teste. Em seguida, criamos os `DataLoaders` respectivos.

In [ ]:
# Load the data
df = pd.read_csv(cfg.LABELS_PATH)

# Preprocessing: Inicialmente mantemos transforms básicos
data_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor()
])

# Stratified Split
train_df, val_df, test_df = dl.get_stratified_splits(
    df, 
    test_size=cfg.TEST_SIZE, 
    val_size=cfg.VAL_SIZE, 
    random_state=cfg.RANDOM_SEED
)

# Instantiate Datasets
train_dataset = dl.ButterflyDataset(df=train_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
val_dataset = dl.ButterflyDataset(df=val_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
test_dataset = dl.ButterflyDataset(df=test_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)

# Instantiate DataLoaders
train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

### 3. Análise do Dataset e Balanceamento de Classes
Verificamos a quantidade de dados por partição e analisamos a distribuição das 75 classes de borboletas no conjunto de treino. Como modelos de *Deep Learning* são sensíveis a desbalanceamentos, compreender esta distribuição guiará a nossa estratégia de Data Augmentation com modelos generativos.

In [ ]:
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}\n")

class_counts = dl.analyze_dataset(
    df=train_df, 
    dataset_name="Treino Original", 
    save_dir=cfg.GRAPH_RESULTS_DIR
)

### 4. Visualização dos Dados
Vamos extrair um batch do DataLoader de treino para visualizar a diversidade e a variação intra-classe presente no dataset.

In [ ]:
# Get a batch of images from the train dataloader
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()

for i, img in enumerate(images):
    img = img.permute(1, 2, 0).numpy()
    label_idx = labels[i].item()
    label_name = train_dataset.classes[label_idx]
    axes[i].imshow(img)
    axes[i].set_title(label_name.capitalize(), fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### 5. Instanciação e Treinamento do Modelo Baseline
Inicializamos a arquitetura CNN, o otimizador e a função de custo (Loss). Em seguida, iniciamos o treinamento por um número fixo de épocas. O script salvará automaticamente os pesos da última época e da época com a melhor *Validation Loss* dentro da pasta `results/models/`.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

CURRENT_SEED = 42 # 42, 100, 200

print(f"\n" + "="*60)
print(f" TREINO BASELINE CNN | SEED: {CURRENT_SEED}")
print("="*60)

def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_training_seed(CURRENT_SEED)

# Criamos sub-pastas para não esmagar os resultados anteriores
seed_models_dir = cfg.CNN_MODELS_DIR / f"seed_{CURRENT_SEED}"
seed_plots_dir = cfg.CNN_PLOTS_DIR / f"seed_{CURRENT_SEED}"
seed_results_dir = cfg.CNN_RESULTS_DIR / f"seed_{CURRENT_SEED}"

seed_models_dir.mkdir(parents=True, exist_ok=True)
seed_plots_dir.mkdir(parents=True, exist_ok=True)
seed_results_dir.mkdir(parents=True, exist_ok=True)

n_classes = len(train_dataset.classes)

model = classifier.BaselineCNN(num_classes=n_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print(f"A iniciar treino da Baseline por {cfg.N_EPOCHS} épocas...")

best_model, history = classifier.train_cnn(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=cfg.N_EPOCHS,
    save_dir=seed_models_dir
)

# Guarda os gráficos de loss na pasta desta seed
mtcs.save_cnn_train(history, seed_plots_dir)

# Análise de candidatos (guardada na pasta dos plots desta seed)
print(f"\nTreino e Teste da Seed {CURRENT_SEED} concluídos com sucesso!")
print(f"Resultados guardados em: {seed_results_dir}")

### 6. Avaliação no Conjunto de Teste
Com o modelo treinado, avaliamos sua performance no conjunto de teste (dados nunca vistos pelo modelo). Extraímos métricas detalhadas como Precision, Recall e F1-Score, além da Matriz de Confusão, para entender onde o modelo baseline acerta e quais espécies de borboletas ele mais confunde.

In [ ]:
# Avaliação do modelo e salvamos a matriz de confusão
resultado_json = mtcs.evaluate_cnn(best_model, test_dataset, test_loader, device=device, results_dir=seed_results_dir)

df_recomendadas = mtcs.analyze_augmentation_candidates(resultado_json, save_dir=seed_plots_dir)

7. Filtro de Sanidade e Hard Sample Mining

Nesta etapa, a CNN original atua como oráculo para avaliar as imagens sintéticas em duas fases:

- **Filtro em Cascata**: Retém apenas as amostras mais realistas (validação Top-2 ou Top-5), garantindo a qualidade visual e o volume de dados necessário.

- **Hard Sample Mining**: Prioriza as imagens aprovadas que causam maior incerteza à rede (menor margem entre a 1ª e a 2ª previsão). Estas amostras "difíceis" são injetadas no treino para forçar a aprendizagem de fronteiras de decisão mais robustas.

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader
from tqdm.auto import tqdm

AUGMENTATION_TYPE = "cVAE" # "cVAE", "GAN", "Diffusion"

paths_map = {
    "cVAE": {
        "cand_csv": cfg.LABELS_DIR / "cvae_candidates_labels.csv",
        "cand_img_dir": cfg.IMG_DIR / "augmented_cvae",
        "models_dir": cfg.CNN_VAE_MODELS_DIR,
        "results_dir": cfg.CNN_VAE_RESULTS_DIR
    },
    "GAN": {
        "cand_csv": cfg.LABELS_DIR / "cgan_candidates_labels.csv",
        "cand_img_dir": cfg.IMG_DIR / "augmented_dcgan",
        "models_dir": cfg.CNN_GAN_MODELS_DIR,
        "results_dir": cfg.CNN_GAN_RESULTS_DIR
    },
    "Diffusion": {
        "cand_csv": cfg.LABELS_DIR / "ddpm_candidates_labels.csv",
        "cand_img_dir": cfg.IMG_DIR / "augmented_diff",
        "models_dir": cfg.CNN_DIFF_MODELS_DIR,
        "results_dir": cfg.CNN_DIFF_RESULTS_DIR
    }
}
target_paths = paths_map[AUGMENTATION_TYPE]

print(f"--- FASE 1: HARD SAMPLE MINING ({AUGMENTATION_TYPE}) ---")

n_classes = len(train_dataset.classes)
oracle_cnn = classifier.BaselineCNN(num_classes=n_classes).to(device)
oracle_cnn.load_state_dict(torch.load(cfg.CNN_MODELS_DIR / 'baseline_best_model.pth', map_location=device))
oracle_cnn.eval()

# Extraindo o Top-5
df_candidates = pd.read_csv(target_paths["cand_csv"])

coluna_origem = 'path' if 'path' in df_candidates.columns else 'filename'

# Extrai o nome da imagem, ignorando caminhos antigos do Windows
df_candidates['clean_filename'] = df_candidates[coluna_origem].apply(
    lambda x: os.path.basename(str(x).replace('\\', '/'))
)

# Reconstrói o caminho absoluto usando IMG_DIR (através do target_paths)
df_candidates['path'] = df_candidates['clean_filename'].apply(
    lambda x: str(target_paths["cand_img_dir"] / x)
)

df_candidates_eval = df_candidates.copy()
df_candidates_eval['filename'] = df_candidates['path']

# Passar img_dir="" porque o 'filename' já contém o caminho completo do IMG_DIR
cand_dataset = classifier.ButterflyDataset(df=df_candidates_eval, img_dir="", transform=data_transform)
cand_loader = DataLoader(cand_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

all_margins, all_preds, all_top5 = [], [], []

with torch.no_grad():
    for inputs, _ in tqdm(cand_loader, desc="Oráculo a avaliar"):
        inputs = inputs.to(device)
        outputs = oracle_cnn(inputs)
        probs = F.softmax(outputs, dim=1)

        # Extraímos o Top-5
        top5_probs, top5_indices = torch.topk(probs, 5, dim=1)

        # A margem de dúvida continua a ser a diferença entre a 1ª e a 2ª escolha
        margin = top5_probs[:, 0] - top5_probs[:, 1]

        all_margins.extend(margin.cpu().numpy())
        all_preds.extend(top5_indices[:, 0].cpu().numpy())
        all_top5.extend(top5_indices.cpu().numpy().tolist())

df_candidates['margin'] = all_margins
df_candidates['pred_class_idx'] = all_preds
df_candidates['top5_preds'] = all_top5

# SISTEMA DE CASCATA
coluna_label = 'label' if 'label' in train_df.columns else train_df.columns[1]
contagem_pandas = train_df[coluna_label].value_counts().to_dict()
TARGET_PER_CLASS = max(contagem_pandas.values())

target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
target_classes = target_df['Classe'].tolist()[:4]

final_selected_data = []

print("\n" + "="*85)
print(f" RELATÓRIO DE MINERAÇÃO SOTA ({AUGMENTATION_TYPE})")
print("="*85)

for class_name in target_classes:
    current_count = contagem_pandas.get(class_name, 0)
    needed = max(0, TARGET_PER_CLASS - current_count)
    if needed == 0: continue

    correct_idx = train_dataset.class_to_idx[class_name]

    # Isolar as 500 imagens geradas para esta classe
    pool_classe = df_candidates[df_candidates['label'] == class_name]


    # Filtro Estrito (Classe está no Top-2)
    class_cands = pool_classe[pool_classe['top5_preds'].apply(lambda x: correct_idx in x[:2])]
    filtro_usado = "Top-2 (Ideal)"

    # Relaxamento (Se o Top-2 não encontrou imagens suficientes, tenta no Top-5)
    if len(class_cands) < needed:
        class_cands = pool_classe[pool_classe['top5_preds'].apply(lambda x: correct_idx in x[:5])]
        filtro_usado = "Top-5"

    # Bypass (Se mesmo no Top-5 a rede não viu a classe, forçamos a entrada)
    if len(class_cands) < needed:
        class_cands = pool_classe
        filtro_usado = "Bypass Total"

    # Ordenamos as imagens validadas pela margem (as mais difíceis primeiro)
    class_cands = class_cands.sort_values(by='margin', ascending=True)
    selected = class_cands.head(needed)

    if len(selected) > 0:
        print(f"{class_name:<18} | Faltam: {needed:<4} | Passaram: {len(selected):<4} [{filtro_usado}]")
        for _, row in selected.iterrows():
            final_selected_data.append({'filename': row['path'], 'label': class_name})
    else:
        print(f"{class_name:<18} | Erro crítico: Sem candidatos disponíveis no CSV.")

if len(final_selected_data) == 0:
    raise ValueError("Nenhuma imagem selecionada!")

df_in_memory = pd.DataFrame(final_selected_data)
sintetico_dataset = dl.ButterflyDataset(df=df_in_memory, img_dir="", transform=data_transform)

augmented_train_dataset = ConcatDataset([train_dataset, sintetico_dataset])
augmented_train_loader = DataLoader(
    augmented_train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nDataset Virtual Criado! Total para Retreino: {len(augmented_train_dataset)} imagens.")

### 8. Retreino da CNN

Esta célula é responsável pelo retreino do classificador com o novo conjunto de dados aumentado. Para garantir o rigor científico, a reprodutibilidade e evitar *Data Leakage*, aplicamos uma inicialização controlada por *Seed*. A cada execução, uma nova rede "limpa" é instanciada e treinada. O modelo é posteriormente avaliado no conjunto de teste original e os resultados (Métricas, Matrizes de Confusão e Gráficos) são guardados em diretórios isolados para permitir o cálculo posterior da Média e do Desvio Padrão.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

CURRENT_SEED = 42 # 100, 200, 300, 400

def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_training_seed(CURRENT_SEED)

n_classes = len(train_dataset.classes)
model_aug = classifier.BaselineCNN(num_classes=n_classes).to(device)
optimizer_aug = optim.Adam(model_aug.parameters(), lr=0.001)
criterion_aug = nn.CrossEntropyLoss()

# Cria pastas únicas para esta Seed
seed_models_dir = target_paths["models_dir"] / f"seed_{CURRENT_SEED}"
seed_results_dir = target_paths["results_dir"] / f"seed_{CURRENT_SEED}"
seed_models_dir.mkdir(parents=True, exist_ok=True)
seed_results_dir.mkdir(parents=True, exist_ok=True)

print(f"A iniciar retreino por {cfg.N_EPOCHS} épocas...")

trained_augmented_cnn, augmented_cnn_history = classifier.train_cnn(
    model=model_aug,
    train_loader=augmented_train_loader,
    val_loader=val_loader,
    optimizer=optimizer_aug,
    criterion=criterion_aug,
    num_epochs=cfg.N_EPOCHS,
    device=device,
    save_dir=seed_models_dir
)

# Guardar gráficos de loss para esta Seed específica
mtcs.save_cnn_train(augmented_cnn_history, seed_results_dir)

print(f"\nTeste da Seed {CURRENT_SEED} concluído com sucesso!")
print(f"Resultados guardados em: {seed_results_dir}")

### 9. Avaliação e Comparação
Avaliamos a rede no conjunto de teste intocado e geramos o JSON de métricas. O uso da diretoria dinâmica garante que cada abordagem generativa terá a sua própria matriz de confusão e registo de métricas finais, facilitando a posterior construção de tabelas comparativas.

In [ ]:
print(f"A avaliar a CNN ({AUGMENTATION_TYPE} Augmentation) no conjunto de teste real...")

# Utilizamos a mesma diretoria dinâmica para guardar o JSON e os gráficos de teste
print(f"\nA avaliar no conjunto de teste real...")
resultado_aug_json = mtcs.evaluate_cnn(
    trained_augmented_cnn, 
    test_dataset, 
    test_loader, 
    device=device, 
    results_dir=seed_results_dir
)

print(f"\nAvaliação concluída. Todos os resultados e gráficos para {AUGMENTATION_TYPE} foram guardados em:")
print(f" {target_paths["results_dir"]}")

### 10. Análise de Impacto Direto (Top 4 Classes Críticas)
Nesta secção, avaliamos o sucesso primário da nossa intervenção generativa. Isolamos as 4 classes com pior desempenho (identificadas na *Baseline*) e comparamos visualmente o seu F1-Score antes e depois da introdução das imagens sintéticas. Este teste A/B permite validar empiricamente se o modelo generativo cumpriu o seu objetivo de resolução do desbalanceamento extremo (*Hard-Example Oversampling*).

In [ ]:
print("\n" + "="*50)
print(f" A GERAR RELATÓRIO COMPARATIVO: BASELINE VS {AUGMENTATION_TYPE}")
print("="*50)

# Caminho de onde guardámos a baseline lá na Célula 6 
BASELINE_CSV_PATH = cfg.CNN_RESULTS_DIR / 'classes_para_augmentar.csv'

df_baseline = pd.read_csv(BASELINE_CSV_PATH)
top_4_classes = df_baseline['Classe'].tolist()[:4]

print(f"Analisando o impacto nas classes alvo: {top_4_classes}")

# 2. Chamar a função de desenho do mtcs.py
vis.plot_augmentation_comparison(
    baseline_csv_path=BASELINE_CSV_PATH,
    new_metrics_json=resultado_aug_json,
    target_classes=top_4_classes,
    save_dir=target_paths["plots_dir"],
    aug_type=AUGMENTATION_TYPE)

### 11. Análise de Fronteira de Decisão e Trade-off
A manipulação da distribuição do *dataset* de treino introduz frequentemente um viés na fronteira de decisão da rede. Para garantir a integridade científica do estudo quantificamos que classes sofreram degradação no F1-Score devido à confusão com os artefactos gerados (ex: desfocagem do VAE) em oposição às classes que mais beneficiaram. Esta visualização divergente é crucial para justificar o uso de arquiteturas mais avançadas (como GANs) em iterações futuras.

In [ ]:
print("\n" + "="*50)
print(f" A GERAR ANÁLISE DE TRADE-OFF: O que ganhámos vs O que perdemos")
print("="*50)

# Caminho para o TXT 
BASELINE_TXT_PATH = cfg.CNN_RESULTS_DIR / 'full_classification_report.txt'

# 1. Extrair os dados antigos 
baseline_metrics_dict = mtcs.parse_classification_report_txt(BASELINE_TXT_PATH)

print(f"Sucesso: Lidas {len(baseline_metrics_dict)} classes do relatório TXT.")

# 2. Chamar a função de gráfico divergente do mtcs.py
mtcs.plot_f1_tradeoff(
    baseline_metrics_dict=baseline_metrics_dict,
    new_metrics_dict=resultado_aug_json, 
    save_dir=target_paths["plots_dir"],
    aug_type=AUGMENTATION_TYPE,
    top_n=10
)